# Notebook 01: Sentinel-2 NDVI + Planet Labs Imagery

Covers:
1. Load credentials from `.env`
2. Planet Labs — search scenes, check key validity
3. Sentinel Hub — fetch S2 L2A composite (requires Sentinel Hub credentials)
4. Compute and visualise NDVI / delta from both sources
5. Side-by-side comparison: Planet 3 m vs Sentinel-2 10 m resolution

In [ ]:
import sys
sys.path.insert(0, '..')

import os
from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

PLANET_KEY = os.environ.get('PLANET_API_KEY', '')
SH_CLIENT  = os.environ.get('SENTINEL_HUB_CLIENT_ID', '')

print('PLANET_API_KEY :', PLANET_KEY[:8] + '...' if PLANET_KEY else 'NOT SET')
print('SENTINEL_HUB   :', SH_CLIENT[:8] + '...' if SH_CLIENT else 'NOT SET (optional)')

## 1. Planet Labs — verify key + search scenes over Tumkur

In [ ]:
from apps.api.services.imagery import PlanetFetcher

# Tumkur industrial area, Karnataka
BBOX = [77.0, 13.25, 77.35, 13.55]  # [min_lon, min_lat, max_lon, max_lat]

planet = PlanetFetcher()  # reads PLANET_API_KEY from env

# Search last 90 days — just to confirm the key works
scenes = planet.search_scenes(
    bbox=BBOX,
    start_date='2026-03-01',
    end_date='2026-06-04',
    item_type='planetscope',
    max_cloud_cover=0.20,
    limit=20,
)

print(f'Found {len(scenes)} PlanetScope scenes')
for s in scenes[:5]:
    p = s['properties']
    print(f"  {s['id']}  acquired={p.get('acquired','?')[:10]}  cloud={p.get('cloud_cover',0)*100:.1f}%")

## 2. Download the best PlanetScope scene and compute NDVI

In [ ]:
# This activates + downloads the SR asset (~2-10 min first time per scene)
# If you want to skip the wait, jump to the Sentinel Hub cells below.

if scenes:
    ps_arr = planet.get_planetscope_composite(
        bbox=BBOX,
        start_date='2026-03-01',
        end_date='2026-06-04',
    )
    ps_ndvi = PlanetFetcher.compute_ndvi(ps_arr)
    print(f'PS array shape: {ps_arr.shape}  NDVI range: [{ps_ndvi.min():.3f}, {ps_ndvi.max():.3f}]')
else:
    print('No scenes found — check bbox / date range / cloud cover threshold')
    ps_arr  = None
    ps_ndvi = None

In [ ]:
if ps_arr is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # True colour: R=band2, G=band1, B=band0
    rgb = ps_arr[:, :, [2, 1, 0]]
    rgb_display = np.power(np.clip(rgb, 0, 0.15) / 0.15, 0.45)
    axes[0].imshow(rgb_display)
    axes[0].set_title('PlanetScope — True Colour (~3 m)', fontsize=11)
    axes[0].axis('off')

    im_ndvi = axes[1].imshow(ps_ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
    axes[1].set_title('PlanetScope — NDVI', fontsize=11)
    axes[1].axis('off')
    fig.colorbar(im_ndvi, ax=axes[1], fraction=0.046, pad=0.04)

    axes[2].hist(ps_ndvi.ravel(), bins=100, color='forestgreen', edgecolor='white', lw=0.3)
    axes[2].axvline(0.3, color='red', ls='--', lw=1, label='vegetation threshold')
    axes[2].set_title('NDVI Distribution', fontsize=11)
    axes[2].set_xlabel('NDVI'); axes[2].set_ylabel('Pixels')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig('01_planet_ndvi.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved 01_planet_ndvi.png')

## 3. Sentinel-2 composite (requires Sentinel Hub credentials)

In [ ]:
from apps.api.services.imagery import SentinelHubFetcher
from apps.api.services.change.generic import compute_ndvi

if not SH_CLIENT or SH_CLIENT == 'your-sentinel-hub-client-id':
    print('Sentinel Hub credentials not set — skipping S2 cells.')
    print('Sign up free at https://www.sentinel-hub.com/ and add to .env')
    s2_before = s2_after = ndvi_before = ndvi_after = ndvi_delta = None
else:
    fetcher = SentinelHubFetcher()
    s2_before = fetcher.get_sentinel2_composite(BBOX, '2025-01-01', '2025-01-31')
    s2_after  = fetcher.get_sentinel2_composite(BBOX, '2026-01-01', '2026-01-31')
    ndvi_before = compute_ndvi(s2_before)
    ndvi_after  = compute_ndvi(s2_after)
    ndvi_delta  = ndvi_after - ndvi_before
    print(f'S2 before: {s2_before.shape}  S2 after: {s2_after.shape}')
    print(f'NDVI delta: mean={ndvi_delta.mean():.4f}  std={ndvi_delta.std():.4f}')

In [ ]:
if s2_before is not None:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    def s2_rgb(arr):
        rgb = arr[:, :, [3, 2, 1]]  # B04=R, B03=G, B02=B
        return np.power(np.clip(rgb, 0, 0.3) / 0.3, 0.4)

    axes[0, 0].imshow(s2_rgb(s2_before))
    axes[0, 0].set_title('S2 Before — True Colour', fontsize=11); axes[0, 0].axis('off')

    axes[0, 1].imshow(ndvi_before, cmap='RdYlGn', vmin=-0.3, vmax=0.8)
    axes[0, 1].set_title('S2 Before — NDVI', fontsize=11); axes[0, 1].axis('off')

    axes[0, 2].imshow(s2_rgb(s2_after))
    axes[0, 2].set_title('S2 After — True Colour', fontsize=11); axes[0, 2].axis('off')

    axes[1, 0].imshow(ndvi_after, cmap='RdYlGn', vmin=-0.3, vmax=0.8)
    axes[1, 0].set_title('S2 After — NDVI', fontsize=11); axes[1, 0].axis('off')

    im = axes[1, 1].imshow(ndvi_delta, cmap='RdBu', vmin=-0.4, vmax=0.4)
    axes[1, 1].set_title('NDVI Delta (After − Before)', fontsize=11); axes[1, 1].axis('off')
    fig.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

    axes[1, 2].hist(ndvi_delta.ravel(), bins=100, color='steelblue', edgecolor='white', lw=0.3)
    axes[1, 2].axvline(0, color='red', ls='--', lw=1)
    axes[1, 2].set_title('NDVI Delta Distribution', fontsize=11)
    axes[1, 2].set_xlabel('NDVI delta'); axes[1, 2].set_ylabel('Pixel count')

    plt.tight_layout()
    plt.savefig('01_s2_ndvi_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Resolution comparison — Planet 3 m vs Sentinel-2 10 m

In [ ]:
if ps_arr is not None and s2_after is not None:
    from skimage.transform import resize  # type: ignore

    h, w = ps_arr.shape[:2]
    s2_resized = resize(s2_rgb(s2_after), (h, w), anti_aliasing=True)
    ps_rgb_disp = np.power(np.clip(ps_arr[:, :, [2, 1, 0]], 0, 0.15) / 0.15, 0.45)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
    ax1.imshow(s2_resized)
    ax1.set_title('Sentinel-2 (~10 m, upsampled to PS resolution)', fontsize=11)
    ax1.axis('off')
    ax2.imshow(ps_rgb_disp)
    ax2.set_title('PlanetScope (~3 m native)', fontsize=11)
    ax2.axis('off')
    plt.suptitle('Tumkur Region — Resolution Comparison', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('01_resolution_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Resolution comparison saved.')
elif ps_arr is not None:
    print('Add Sentinel Hub credentials to .env to generate the comparison plot.')
else:
    print('No Planet data available — run cells 3-6 first.')

## 5. Summary statistics

In [ ]:
import pandas as pd

rows = []
for name, arr in [('Planet NDVI', ps_ndvi), ('S2 NDVI delta', ndvi_delta)]:
    if arr is None:
        continue
    rows.append({
        'Source': name,
        'Min':  f'{arr.min():.4f}',
        'Max':  f'{arr.max():.4f}',
        'Mean': f'{arr.mean():.4f}',
        'Std':  f'{arr.std():.4f}',
        'Vegetation (>0.3)': f'{(arr > 0.3).mean()*100:.1f}%',
    })

if rows:
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print('Run at least one of the imagery cells above first.')